In [1]:
%pip install -q transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import transformers
import torch

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers version: 5.21.0
PyTorch version: 2.14.1+cpu
CUDA available: False


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Tên mô hình trên Hugging Face Hub
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# Tải tokenizer tương ứng với mô hình
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tải mô hình đã được huấn luyện cho phân loại cảm xúc
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Chuyển sang chế độ đánh giá / dự đoán
model.eval()

print("Đã tải tokenizer và mô hình thành công!")
print("Các nhãn:", model.config.id2label)

d:\learn_DeepLearning\Lab3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\learn_DeepLearning\Lab3\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlink

Đã tải tokenizer và mô hình thành công!
Các nhãn: {0: 'NEGATIVE', 1: 'POSITIVE'}


In [1]:
import sys
import torch
import transformers
from transformers.utils import is_torch_available

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Transformers nhận diện PyTorch:", is_torch_available())

d:\learn_DeepLearning\Lab3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: d:\learn_DeepLearning\Lab3\.venv\Scripts\python.exe
PyTorch: 2.14.1+cpu
Transformers: 5.21.0
Transformers nhận diện PyTorch: True


In [3]:
sentence = "I love this movie. It is amazing!"

# Chuyển câu thành đầu vào dạng tensor của PyTorch
inputs = tokenizer(
    sentence,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Đổi các mã số về token để quan sát
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

print("Câu gốc:", sentence)
print("\nCác token:", tokens)
print("\nInput IDs:", inputs["input_ids"])
print("\nAttention mask:", inputs["attention_mask"])
print("\nKích thước đầu vào:", inputs["input_ids"].shape)

Câu gốc: I love this movie. It is amazing!

Các token: ['[CLS]', 'i', 'love', 'this', 'movie', '.', 'it', 'is', 'amazing', '!', '[SEP]']

Input IDs: tensor([[ 101, 1045, 2293, 2023, 3185, 1012, 2009, 2003, 6429,  999,  102]])

Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Kích thước đầu vào: torch.Size([1, 11])


In [4]:
import torch

model.eval()

# Chỉ dự đoán, không tính gradient
with torch.no_grad():
    outputs = model(**inputs)

# Chuyển điểm logits thành xác suất của hai lớp
probabilities = torch.softmax(outputs.logits, dim=-1)

# Chọn lớp có xác suất lớn nhất
predicted_id = probabilities.argmax(dim=-1).item()
predicted_label = model.config.id2label[predicted_id]
confidence = probabilities[0, predicted_id].item()

print("Câu:", sentence)
print("\nLogits:", outputs.logits)

print("\nXác suất từng lớp:")
for class_id, probability in enumerate(probabilities[0]):
    label = model.config.id2label[class_id]
    print(f"  {label}: {probability.item():.4%}")

print("\nCảm xúc dự đoán:", predicted_label)
print(f"Điểm tin cậy: {confidence:.4%}")

Câu: I love this movie. It is amazing!

Logits: tensor([[-4.3436,  4.7032]])

Xác suất từng lớp:
  NEGATIVE: 0.0118%
  POSITIVE: 99.9882%

Cảm xúc dự đoán: POSITIVE
Điểm tin cậy: 99.9882%
